# Description

In this notebook, I will prepare the dataset with if statement.

In [1]:
import ast, textwrap, random, os
import pandas as pd
import numpy as np
import json
from typing import List, Optional, Tuple, Dict
from dataclasses import dataclass
import torch
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizerFast, DataCollatorWithPadding
from sklearn.model_selection import train_test_split

In [2]:
def _find_first_if_condition_span(src: str) -> Optional[Tuple[int, int, str]]:
    """
    Returns (start_char, end_char, condition_text) for the first if/elif condition.
    Requires Python 3.8+ for end_lineno/end_col_offset. Skips if not available.
    """
    try:
        tree = ast.parse(src)
        lines = src.splitlines(keepends=True)

        def to_abs(lineno, col, end_lineno, end_col):
            start = sum(len(l) for l in lines[:lineno-1]) + col
            end   = sum(len(l) for l in lines[:end_lineno-1]) + end_col
            return start, end

        for node in ast.walk(tree):
            if isinstance(node, ast.If):
                test = node.test
                if hasattr(test, "lineno") and hasattr(test, "end_lineno"):
                    s, e = to_abs(test.lineno, test.col_offset,
                                  test.end_lineno, test.end_col_offset)
                    cond_text = src[s:e]
                    return s, e, cond_text
        return None
    except Exception:
        return None

def _mask_once(src: str, mask_token: str) -> Optional[Tuple[str, str]]:
    """
    Replace exactly one if-condition in `src` with `mask_token`.
    Returns (masked_function_text, original_condition_text) or None if not found.
    """
    span = _find_first_if_condition_span(src)
    if not span:
        return None
    s, e, cond = span
    if not cond.strip():
        return None
    masked = src[:s] + mask_token + src[e:]
    return masked, cond

def prepare_if_mask_dataset(
    functions: List[str],
    tok,
    out_dir: str = "if_dataset",
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42,
) -> None:
    """
    Build a dataset of (input=function_with_one_masked_if_condition, target=condition_text)
    and save to JSONL files: train.jsonl, validation.jsonl, test.jsonl.
    """
    
    os.makedirs(out_dir, exist_ok=True)

    mask_token = tok.mask_token

    # Shuffle once for splits
    rng = random.Random(seed)
    idxs = list(range(len(functions)))
    rng.shuffle(idxs)

    # Build examples
    records = []
    dropped = 0
    for i in idxs:
        src = functions[i]
        out = _mask_once(src, mask_token)
        if out is None:
            dropped += 1
            continue
        masked, cond = out
        records.append({
            "id": i,
            "input": masked,
            "target": cond.strip()
        })

    if not records:
        raise ValueError("No usable functions with if-conditions were found.")

    # Split train/val/test
    n = len(records)
    train_val, test = train_test_split(records, test_size=test_ratio, random_state=seed)
    train, val = train_test_split(train_val, test_size=val_ratio, random_state=seed)

    # Save JSONL
    def write_jsonl(path, rows):
        with open(path, "w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    write_jsonl(os.path.join(out_dir, "train.jsonl"), train)
    write_jsonl(os.path.join(out_dir, "validation.jsonl"), val)
    write_jsonl(os.path.join(out_dir, "test.jsonl"), test)

    # Save some metadata
    manifest = {
        "total_kept": n,
        "dropped_no_if": dropped,
        "mask_token": mask_token,
        "splits": {"train": len(train), "validation": len(val), "test": len(test)}
    }
    with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print(f"[DONE] Saved to {out_dir}")
    print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [3]:
PATH_DATA_FILE = os.path.join(os.getcwd(), 'dataset', 'processed', 'processed_data.csv')
PATH_TOKENIZER_FILE = os.path.join(os.getcwd(), 'python_tokenizer.json')

In [4]:
df = pd.read_csv(PATH_DATA_FILE)
list_python_function = df['method_code'].tolist()
print(f"Total functions: {len(list_python_function)}")

Total functions: 1371223


In [5]:
tok = PreTrainedTokenizerFast(tokenizer_file=PATH_TOKENIZER_FILE)
tok.add_special_tokens({
    "pad_token": "<pad>", "unk_token": "<unk>", "mask_token": "<mask>",
    "bos_token": "<s>", "eos_token": "</s>",
})
vocab_size = len(tok)

In [6]:
prepare_if_mask_dataset(
    functions=list_python_function,
    tok=tok,
    out_dir="if_dataset",
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
)

[DONE] Saved to if_dataset
{
  "total_kept": 412300,
  "dropped_no_if": 958923,
  "mask_token": "<mask>",
  "splits": {
    "train": 333963,
    "validation": 37107,
    "test": 41230
  }
}
